# Exploración: OSCE - Registro Nacional de Proveedores (RNP)

- **Entidad oficial:** Organismo Supervisor de las Contrataciones del Estado (OSCE) — Registro Nacional de Proveedores (RNP).
- **Descripción y propósito:** Padrón oficial que acredita y habilita a personas naturales y jurídicas para contratar con el Estado peruano. Permite verificar el historial de trámites de inscripción y reinscripción del proveedor, así como cruzar fechas exactas de pago y números de expediente para detectar vinculaciones o trámites correlativos entre empresas competidoras.
- **Enlace de consulta:** `https://www.rnp.gob.pe/Constancia/RNP_Constancia/ConsultaRuc.asp`
- **Criterio de búsqueda:** Exclusivamente por **RUC (11 dígitos)** o Código de Extranjero (no admite consultas directas por DNI de 8 dígitos).
- **Información que se obtiene:**
  - Datos de identificación del proveedor: Número de RUC y Nombre o Razón Social.
  - Historial de inscripciones por capítulo (`BIENES` y `SERVICIOS`).
  - Código de proveedor asignado (`B...` para bienes y `S...` para servicios).
  - Fecha exacta de pago de la tasa y Número de trámite / cargo (`# TRAMITE/CARGO`).
  - Estado de la inscripción (ej. `APROBACION AUTOMATICA`, `SOLICITUD DE DEVOLUCION PROCEDE`) y motivo del estado (`REQUISITOS CONFORME`, etc.).

## Estrategia

- **Arquitectura del portal:** Aplicación monolítica heredada en **ASP Clásico (`ConsultaRuc.asp`)** sobre Microsoft IIS. No expone endpoints REST/JSON en segundo plano; al enviar el formulario ejecuta un `POST` tradicional (`?action=enviar`) que recarga el documento completo e incrusta las tablas HTML de resultados en la misma vista.
- **Resolución automatizada de CAPTCHA:**
  1. **Captura en sesión activa:** Con `Playwright`, se toma una captura en memoria (`screenshot()`) directamente del nodo `<img>` del CAPTCHA ya renderizado en el DOM, garantizando que el código coincida con la cookie `ASPSESSIONID` actual sin disparar una nueva generación en el servidor.
  2. **Limpieza morfológica (`Pillow`):** Se escala la imagen (`x3`), se binariza mediante múltiples umbrales (`135`, `115`, `155`), se aplica un filtro de mediana (`MedianFilter`) para eliminar la línea transversal de ruido de 1 píxel y se añade un margen blanco (*padding*) de `20px`.
  3. **Reconocimiento óptico (`Tesseract.js` in-memory):** Se evalúa la imagen preprocesada en un worker de `Tesseract.js` restringido a caracteres alfanuméricos en mayúsculas (`A-Z0-9`).
  4. **Validación previa al envío:** Como el código del RNP consta siempre de **6 caracteres**, cualquier lectura de longitud distinta se descarta en local y acciona el enlace *"Refrescar código"* antes de enviar el `POST`, optimizando tiempos de respuesta.
- **Parseo estructurado (`BeautifulSoup`):** Recorrido secuencial de filas `<tr>` identificando las cabeceras divisoras (`INSCRIPCION EN EL RNP - BIENES` / `SERVICIOS`) para agrupar dinámicamente cada bloque de trámite dentro de una lista normalizada.

In [8]:
import asyncio
import base64
from io import BytesIO
from PIL import Image, ImageOps, ImageFilter
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re
import json

RUC_PRUEBA = "20607524476"
URL_RNP = "https://www.rnp.gob.pe/Constancia/RNP_Constancia/ConsultaRuc.asp"

def limpiar(texto: str) -> str:
    if not texto:
        return ""
    return re.sub(r"\s+", " ", texto).strip()

def preprocesar_captcha_rnp(img_bytes: bytes, umbral: int = 135) -> str:
    # 1. Convertir a escala de grises y recortar bordes oscuros si existen
    img = Image.open(BytesIO(img_bytes)).convert("L")
    
    # 2. Aumentar resolución x3 para que el filtro de mediana elimine la línea delgada sin borrar las letras gruesas
    img = img.resize((img.width * 3, img.height * 3), Image.Resampling.BICUBIC)
    
    # 3. Binarizar y aplicar MedianFilter para borrar la raya diagonal de 1px
    img = img.point(lambda p: 255 if p > umbral else 0)
    img = img.filter(ImageFilter.MedianFilter(size=3))
    
    # 4. Agregar margen blanco (padding) de 20px alrededor: crítico para que Tesseract no falle en la primera y última letra
    img = ImageOps.expand(img, border=20, fill=255)
    
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"

def parsear_resultado_rnp(html: str) -> dict | None:
    soup = BeautifulSoup(html, "html.parser")
    texto_completo = limpiar(soup.get_text()).upper()

    # Caso 1: El RUC no tiene inscripción en RNP (mensaje explícito del portal)
    if "NO SE ENCONTRO INFORMACION" in texto_completo or "NO REGISTRA" in texto_completo:
        return {"ruc": None, "nombre": None, "inscripciones": [], "sin_registros": True}

    # Caso 2: El captcha falló y la tabla de resultados no se renderizó
    if "INSCRIPCION EN EL RNP" not in texto_completo and "PROVEEDOR" not in texto_completo:
        return None

    resultado = {
        "ruc": None,
        "nombre": None,
        "inscripciones": []
    }

    inscripcion_actual = None

    for tr in soup.find_all("tr"):
        celdas = tr.find_all(["td", "th"])
        if not celdas:
            continue

        texto_fila = limpiar(tr.get_text()).upper()

        # Detectar cabecera de capítulo (ej: "INSCRIPCION EN EL RNP - BIENES" o "SERVICIOS")
        if "INSCRIPCION EN EL RNP" in texto_fila and len(celdas) == 1:
            if inscripcion_actual:
                resultado["inscripciones"].append(inscripcion_actual)
            partes = texto_fila.split("-")
            capitulo = partes[-1].strip() if len(partes) > 1 else texto_fila
            inscripcion_actual = {
                "capitulo": capitulo,
                "codigo_proveedor": None,
                "fecha_pago": None,
                "numero_tramite": None,
                "estado_inscripcion": None,
                "motivo": None
            }
            continue

        # Filas de clave-valor (pueden tener 2 celdas o celdas anidadas)
        if len(celdas) >= 2:
            etiqueta = limpiar(celdas[0].get_text()).upper()
            valor = limpiar(celdas[1].get_text())

            if etiqueta.startswith("RUC"):
                resultado["ruc"] = valor
            elif etiqueta.startswith("NOMBRE"):
                resultado["nombre"] = valor
            elif inscripcion_actual is not None:
                if "PROVEEDOR" in etiqueta:
                    inscripcion_actual["codigo_proveedor"] = valor
                elif "PAGO" in etiqueta:
                    inscripcion_actual["fecha_pago"] = valor
                elif "TRAMITE" in etiqueta or "CARGO" in etiqueta:
                    inscripcion_actual["numero_tramite"] = valor
                elif "ESTADO" in etiqueta:
                    inscripcion_actual["estado_inscripcion"] = valor
                elif "MOTIVO" in etiqueta:
                    inscripcion_actual["motivo"] = valor

    if inscripcion_actual:
        resultado["inscripciones"].append(inscripcion_actual)

    return resultado

async def consultar_rnp(ruc: str, max_intentos: int = 12) -> dict:
    async with async_playwright() as p:
        try:
            browser = await p.chromium.launch(headless=True, channel="chrome")
        except Exception:
            browser = await p.chromium.launch(headless=True, channel="msedge")

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        # Pestaña dedicada con worker Tesseract.js precalentado (PSM 7 = línea de texto única)
        ocr_page = await context.new_page()
        await ocr_page.set_content("""
            <script src="https://cdn.jsdelivr.net/npm/tesseract.js@5/dist/tesseract.min.js"></script>
            <script>
                window.workerReady = (async () => {
                    const worker = await Tesseract.createWorker('eng');
                    await worker.setParameters({
                        tessedit_char_whitelist: 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                        tessedit_pageseg_mode: '7'
                    });
                    return worker;
                })();
                window.leerCaptcha = async (dataUrl) => {
                    const worker = await window.workerReady;
                    const { data: { text } } = await worker.recognize(dataUrl);
                    return text;
                };
            </script>
        """)

        print(f"[1/3] Conectando a RNP ({URL_RNP})...")
        await page.goto(URL_RNP, wait_until="domcontentloaded", timeout=30000)

        for intento in range(1, max_intentos + 1):
            input_ruc = page.locator("input[type='text']").first
            input_captcha = page.locator("input[type='text']").nth(1)
            img_captcha = page.locator("img[src*='captcha' i], img[src*='codigo' i], form img").first

            await input_ruc.fill(ruc)
            captcha_bytes = await img_captcha.screenshot()

            # Probar con dos umbrales distintos en memoria si el primero no arroja los 6 caracteres exactos
            codigo_resuelto = ""
            for umbral in (135, 115, 155):
                img_b64 = preprocesar_captcha_rnp(captcha_bytes, umbral=umbral)
                texto_crudo = await ocr_page.evaluate("dataUrl => window.leerCaptcha(dataUrl)", img_b64)
                candidato = re.sub(r"[^A-Z0-9]", "", texto_crudo.upper())
                if len(candidato) == 6:
                    codigo_resuelto = candidato
                    break

            print(f"[2/3] Intento {intento}/{max_intentos} -> Captcha 6-chars: '{codigo_resuelto}'")

            # El captcha de RNP SIEMPRE tiene 6 caracteres; si no tiene 6, refrescar sin perder tiempo en el POST
            if len(codigo_resuelto) != 6:
                link_refrescar = page.locator("a:has-text('Refrescar')").first
                if await link_refrescar.count() > 0:
                    await link_refrescar.click()
                    await page.wait_for_timeout(350)
                continue

            await input_captcha.fill(codigo_resuelto)

            btn_buscar = page.locator("input[value='Buscar'], button:has-text('Buscar')").first
            async with page.expect_navigation(wait_until="domcontentloaded", timeout=15000):
                await btn_buscar.click()

            html_actual = await page.content()
            datos = parsear_resultado_rnp(html_actual)

            if datos:
                if datos.get("sin_registros"):
                    print(f"[3/3] El RUC {ruc} no registra trámites en el RNP.")
                    await browser.close()
                    return {"ruc": ruc, "nombre": None, "inscripciones": []}
                if datos.get("ruc"):
                    print(f"[3/3] ¡Extracción exitosa en el intento {intento}! ({len(datos['inscripciones'])} registros encontrados)")
                    await browser.close()
                    return datos

            print(" -> El captcha fue rechazado por el servidor. Probando nuevo código...")

        await browser.close()
        return {"error": f"No se pudo superar el captcha tras {max_intentos} intentos."}

# Ejecución en el notebook
resultado_rnp = await consultar_rnp(RUC_PRUEBA)
print("\n================== RESULTADO RNP EXTRAÍDO ==================")
print(json.dumps(resultado_rnp, indent=2, ensure_ascii=False))

[1/3] Conectando a RNP (https://www.rnp.gob.pe/Constancia/RNP_Constancia/ConsultaRuc.asp)...
[2/3] Intento 1/12 -> Captcha 6-chars: 'GMULTH'
 -> El captcha fue rechazado por el servidor. Probando nuevo código...
[2/3] Intento 2/12 -> Captcha 6-chars: '672173'
[3/3] ¡Extracción exitosa en el intento 2! (2 registros encontrados)

================== RESULTADO RNP EXTRAÍDO ==================
{
  "ruc": "20607524476",
  "nombre": "INVERSIONES Y SERVICIOS HUAYNA DEL PERU EMPRESA INDIVIDUAL DE RESPONSABILIDAD LIMITADA",
  "inscripciones": [
    {
      "capitulo": "BIENES",
      "codigo_proveedor": null,
      "fecha_pago": "18/03/2021",
      "numero_tramite": "2021-18895585",
      "estado_inscripcion": "APROBACION AUTOMATICA",
      "motivo": "REQUISITOS CONFORME"
    },
    {
      "capitulo": "SERVICIOS",
      "codigo_proveedor": null,
      "fecha_pago": "18/03/2021",
      "numero_tramite": "2021-18895643",
      "estado_inscripcion": "APROBACION AUTOMATICA",
      "motivo": "REQUISI